# 04 — Model Explainability & Visual Diagnostics
**AutoDS Scientific Methodology Demonstration**

This notebook demonstrates feature importance extraction, SHAP attributions, and dataset-agnostic visual diagnostic generation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backend.app.tools.preprocessor import prepare_train_test_split
from backend.app.tools.ml_trainer import train_and_evaluate_model, evaluate_locked_champion_on_holdout
from backend.app.tools.explainability import calculate_feature_importance
from backend.app.tools.visualizer import (
    generate_roc_pr_plots,
    generate_confusion_matrix_plot,
    generate_feature_importance_plot
)

df = pd.read_csv("data/raw/31513e04_winequality-red.csv", sep=None, engine="python")
X_train, X_test, y_train, y_test, prep = prepare_train_test_split(df, target_column="quality", problem_type="classification", test_size=0.2, random_state=42)
exp = train_and_evaluate_model("LogisticRegression", "classification", X_train, y_train, feature_names=prep.feature_names, cv_folds=3, track_mlflow=False)
evaluate_locked_champion_on_holdout(exp, X_train, y_train, X_test, y_test, track_mlflow=False)
print("Champion model ready for explainability extraction.")


Champion model ready for explainability extraction.



## 1. Top Predictive Drivers (Relative Importance)
Note: These rankings reflect model-derived statistical signal in observational data and do not establish causal mechanisms.


In [2]:
feat_imp = calculate_feature_importance(exp["model"], prep.feature_names)
print(f"Top 5 Predictive Drivers for {exp['model_name']}:")
for r in feat_imp["rankings"][:5]:
    print(f"  - {r['feature']:<25}: {r['importance_pct']:.2f}%")


Top 5 Predictive Drivers for LogisticRegression:
  - volatile acidity         : 4.74%
  - alcohol                  : 4.55%
  - density                  : 3.22%
  - total sulfur dioxide     : 2.89%
  - sulphates                : 2.22%



## 2. Generating Core Classification Visual Diagnostics
We generate and display the 4 core diagnostic figures: Multiclass OvR ROC Curve, Precision-Recall Curve, Confusion Matrix, and Feature Importance.


In [3]:
test_m = exp["metrics"]["test"]
roc_pr = generate_roc_pr_plots(
    roc_data=test_m["roc_curve"],
    pr_data=test_m["pr_curve"],
    roc_auc=test_m["roc_auc"],
    pr_auc=test_m["pr_auc"],
    model_name=exp["model_name"],
    run_id="nb_demo"
)

cm_path = generate_confusion_matrix_plot(
    cm=test_m["confusion_matrix"],
    model_name=exp["model_name"],
    run_id="nb_demo",
    class_labels=test_m["class_labels"]
)

imp_path = generate_feature_importance_plot(
    feature_rankings=feat_imp["rankings"],
    model_name=exp["model_name"],
    run_id="nb_demo"
)

print(f"Generated ROC Path:     {roc_pr['roc_curve_path']}")
print(f"Generated PR Path:      {roc_pr['pr_curve_path']}")
print(f"Generated CM Path:      {cm_path}")
print(f"Generated Feature Path: {imp_path}")


Generated ROC Path:     reports/artifacts/nb_demo_LogisticRegression_roc.png
Generated PR Path:      reports/artifacts/nb_demo_LogisticRegression_pr.png
Generated CM Path:      reports/artifacts/nb_demo_LogisticRegression_cm.png
Generated Feature Path: reports/artifacts/nb_demo_LogisticRegression_feature_imp.png

